## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal([28*28, 128]), name='W1')
        self.b1 = tf.Variable(tf.zeros([128]), name='b1')
        self.W2 = tf.Variable(tf.random.normal([128, 10]), name='W2')
        self.b2 = tf.Variable(tf.zeros([10]), name='b2')
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(x, [-1, 28*28])
        h = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h, self.W2) + self.b2
        ####################
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    # 使用optimizer更新参数
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()

# 将数据转换为float32
train_x = tf.constant(train_data[0], dtype=tf.float32)
train_y = tf.constant(train_data[1], dtype=tf.int64)
test_x = tf.constant(test_data[0], dtype=tf.float32)
test_y = tf.constant(test_data[1], dtype=tf.int64)

# 使用batch训练
train_dataset = tf.data.Dataset.from_tensor_slices((train_x, train_y))
train_dataset = train_dataset.shuffle(60000).batch(128)

for epoch in range(10):
    for step, (x_batch, y_batch) in enumerate(train_dataset):
        loss, accuracy = train_one_step(model, optimizer, x_batch, y_batch)
        if step % 100 == 0:
            print('epoch', epoch, ', step', step, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
    
    # 每个epoch结束后测试
    loss, accuracy = test(model, test_x, test_y)
    print(f'epoch {epoch} test: loss {loss.numpy()}, accuracy {accuracy.numpy()}')

# 最终测试
loss, accuracy = test(model, test_x, test_y)
print('final test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 , step 0 : loss 93.94278 ; accuracy 0.1015625
epoch 0 , step 100 : loss 23.48151 ; accuracy 0.4609375
epoch 0 , step 200 : loss 10.151764 ; accuracy 0.6484375
epoch 0 , step 300 : loss 7.6674585 ; accuracy 0.6875
epoch 0 , step 400 : loss 9.408914 ; accuracy 0.7109375
epoch 0 test: loss 6.435498237609863, accuracy 0.786899983882904
epoch 1 , step 0 : loss 6.552973 ; accuracy 0.7578125
epoch 1 , step 100 : loss 4.612958 ; accuracy 0.8359375
epoch 1 , step 200 : loss 4.25194 ; accuracy 0.828125
epoch 1 , step 300 : loss 4.03888 ; accuracy 0.8203125
epoch 1 , step 400 : loss 4.415102 ; accuracy 0.828125
epoch 1 test: loss 3.900989055633545, accuracy 0.8539999723434448
epoch 2 , step 0 : loss 2.6453457 ; accuracy 0.8828125
epoch 2 , step 100 : loss 3.5592227 ; accuracy 0.84375
epoch 2 , step 200 : loss 3.1590374 ; accuracy 0.859375
epoch 2 , step 300 : loss 1.714525 ; accuracy 0.9296875
epoch 2 , step 400 : loss 4.2001886 ; accuracy 0.8515625
epoch 2 test: loss 2.8894753456115723, 